# 02 Audit: data leakage and error analysis

Runs on CPU (no GPU, no Internet). Inputs:
1. The cache dataset (`X224.npy`, `meta.csv`, `calidad.csv`) produced by `00_preprocessing_cache.ipynb`.
2. A dataset with the folders `resultados/<model>/` of the six models (`pred_test_*.csv`, `metrics_*.json`, `history_*.csv`).

It reads the whole cache (about 6 GB), so it takes about 15 minutes. Tables and figures are saved in `/kaggle/working/auditoria/`.
Sections: 1 integrity, 2 descriptors, 3 duplicates, 4 spatial overlap, 5 effect of the overlap (Fisher) and 5b within classes (Mantel-Haenszel),
6 accuracy by loss range, 7 ordinal metrics, 8 classical baseline, 9 image gaps, 10 training dynamics, 11 agreement between models,
12 accuracy by zone, 13 summary, 15 formulations of the Random Forest (training and validation only).

In [ ]:
# Cell 1: load everything
import os, glob, json, hashlib, numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy import stats as ss
from sklearn.metrics import accuracy_score, f1_score, cohen_kappa_score, mean_absolute_error
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40)
INPUT, OUTD = "/kaggle/input", "/kaggle/working/auditoria"; os.makedirs(OUTD, exist_ok=True)
find = lambda pat: sorted(glob.glob(f"{INPUT}/**/{pat}", recursive=True))
DIR = os.path.dirname(find("X224.npy")[0]); print("cache:", DIR)
meta = pd.read_csv(f"{DIR}/meta.csv"); cal = pd.read_csv(f"{DIR}/calidad.csv")
X = np.load(f"{DIR}/X224.npy", mmap_mode="r"); print("X:", X.shape)
CL = ["No deforestation", "Low", "Moderate", "Severe"]
PRED, MET, HIST = {}, {}, {}
test_cache = set(meta.loc[meta.split == "test", "patch_id"])
for f in find("pred_test_*.csv"):
    m = os.path.basename(f)[len("pred_test_"):-4]; p = pd.read_csv(f); d = os.path.dirname(f)
    if m in PRED:
        if set(PRED[m].patch_id) == test_cache or set(p.patch_id) != test_cache:
            print("warning, repeated version of", m, "ignored:", f); continue
        print("warning,", m, "replaced by the version that uses the test subset of the cache:", f)
    PRED[m] = p
    if os.path.exists(f"{d}/metrics_{m}.json"): MET[m] = json.load(open(f"{d}/metrics_{m}.json"))
    if os.path.exists(f"{d}/history_{m}.csv"): HIST[m] = pd.read_csv(f"{d}/history_{m}.csv")
print("models found:", list(PRED))

In [ ]:
# Cell 2: 1. integrity of the partition and of the labels
print("== 1. Integrity ==")
print("unique patch_id:", meta.patch_id.is_unique)
print(pd.crosstab(meta.label, meta.split, margins=True))
def etiqueta(x): return 0 if x == 0 else 1 if x <= 3 else 2 if x <= 7 else 3
print("labels inconsistent with loss_pct and the thresholds 0/3/7:", int((meta.loss_pct.apply(etiqueta) != meta.label).sum()))
test_ids = set(meta.loc[meta.split == "test", "patch_id"])
OK = []
for m, p in PRED.items():
    s = set(p.patch_id); igual = s == test_ids
    print(f"  {m:16s} rows={len(p):4d} | same test as the cache: {igual} | patches in common: {len(s & test_ids)}")
    if igual: OK.append(m)
print("comparable models (same test):", OK)
print("models EXCLUDED from the analysis:", [m for m in PRED if m not in OK])

In [ ]:
# Cell 3: 2. single reading of the cache (hash, visual signature and descriptors of each patch)
feats, firmas, hashes = [], [], []
for i in range(len(meta)):
    raw = np.asarray(X[i]); hashes.append(hashlib.md5(raw.tobytes()).hexdigest())
    a = raw.astype(np.float32)                                   # (5, 8, S, S), without normalization
    k = a.shape[-1] // 14
    f = a[4, :6, :14 * k, :14 * k].reshape(6, 14, k, 14, k).mean((2, 4)).ravel()   # 2024 signature
    firmas.append((f - f.mean()) / (f.std() + 1e-6))
    nd = a[:, 6]; dnd = nd[4] - nd[0]
    d = {}
    for t in (0, 4):
        for c in range(8):
            v = a[t, c]; d[f"m{t}_{c}"], d[f"s{t}_{c}"] = v.mean(), v.std()
            d[f"p10_{t}_{c}"], d[f"p90_{t}_{c}"] = np.percentile(v, [10, 90])
    for c in range(8): d[f"dm_{c}"] = a[4, c].mean() - a[0, c].mean()
    d["dndvi_medio"] = dnd.mean(); d["frac_caida_ndvi"] = (dnd < -0.2).mean()
    d["frac_suelo_2024"] = (nd[4] < 0.5).mean(); d["ndvi_min_serie"] = nd.mean((1, 2)).min()
    feats.append(d)
    if i % 300 == 0: print("read", i)
F = pd.DataFrame(feats); firmas = np.stack(firmas); meta["hash"] = hashes
print("descriptors:", F.shape)                                   # (1500, 76)

In [ ]:
# Cell 4: 3. exact and near duplicates
print("== 3. Duplicates ==")
dup = meta[meta.duplicated("hash", keep=False)]
print("patches with an image identical to another patch:", len(dup))
if len(dup): print(dup[["patch_id", "label", "split", "hash"]].sort_values("hash").head(30).to_string())
idx = {s: np.where(meta.split.values == s)[0] for s in ["train", "val", "test"]}
sim = firmas[idx["test"]] @ firmas[idx["train"]].T / firmas.shape[1]      # Pearson correlation
meta["sim_max_train"] = np.nan
meta.loc[meta.index[idx["test"]], "sim_max_train"] = sim.max(1)
print("maximum visual similarity test-train (1 = identical):")
print(pd.Series(sim.max(1)).describe().round(3).to_string())
print("test patches with similarity > 0.95 with some training patch:", int((sim.max(1) > 0.95).sum()))
print("Note: this coarse signature is similar for most forest patches, so it does not discriminate leakage (see cells 5 and 6).")

In [ ]:
# Cell 5: 4. spatial overlap between test and training patches
print("== 4. Spatial overlap ==")
lons = np.sort(meta.lon_min.unique()); pasos = np.diff(lons)
print("patch size (degrees):", round(float((meta.lon_max - meta.lon_min).median()), 4),
      "| minimum grid step (degrees):", round(float(pasos[pasos > 1e-6].min()), 4))
tr = meta.iloc[idx["train"]]; te = meta.iloc[idx["test"]]
cxt, cyt = (tr.lon_min + tr.lon_max) / 2, (tr.lat_min + tr.lat_max) / 2
sol, nsol, dist = [], [], []
for _, r in te.iterrows():
    w = np.minimum(r.lon_max, tr.lon_max) - np.maximum(r.lon_min, tr.lon_min)
    h = np.minimum(r.lat_max, tr.lat_max) - np.maximum(r.lat_min, tr.lat_min)
    fr = np.clip(w, 0, None) * np.clip(h, 0, None) / ((r.lon_max - r.lon_min) * (r.lat_max - r.lat_min))
    sol.append(fr.max()); nsol.append(int((fr > 0).sum()))
    cx, cy = (r.lon_min + r.lon_max) / 2, (r.lat_min + r.lat_max) / 2
    dist.append(np.sqrt(((cxt - cx) * 111.32 * np.cos(np.radians(cy))) ** 2 + ((cyt - cy) * 110.57) ** 2).min())
meta.loc[te.index, "solape_max"] = sol; meta.loc[te.index, "n_train_solapados"] = nsol
meta.loc[te.index, "dist_km"] = dist
T = meta.loc[te.index]
print("test patches that overlap with some training patch:", int((T.solape_max > 0).sum()), "of", len(T))
print("maximum overlapped fraction:"); print(T.solape_max.describe().round(3).to_string())
print("distance to the nearest training patch (km):"); print(T.dist_km.describe().round(2).to_string())
T[["patch_id", "label", "solape_max", "n_train_solapados", "dist_km", "sim_max_train"]].rename(columns={
    "solape_max": "max_overlap", "n_train_solapados": "n_train_overlapping"}).to_csv(f"{OUTD}/test_overlap.csv", index=False)

In [ ]:
# Cell 6: 5. effect of the overlap on each model (with leakage, accuracy would be higher on overlapping patches)
print("== 5. Accuracy by overlap ==")
filas = []
for m in OK:
    p = PRED[m].merge(T[["patch_id", "solape_max", "sim_max_train"]], on="patch_id"); ok = p.y_true == p.y_pred
    g = p.solape_max > 0
    fila = dict(model=m, acc_total=ok.mean(), n_with_overlap=int(g.sum()), acc_with_overlap=ok[g].mean() if g.any() else np.nan,
                n_without_overlap=int((~g).sum()), acc_without_overlap=ok[~g].mean() if (~g).any() else np.nan)
    if g.any() and (~g).any():
        fila["p_fisher"] = ss.fisher_exact([[ok[g].sum(), (~ok[g]).sum()], [ok[~g].sum(), (~ok[~g]).sum()]])[1]
    fila["rho_similarity_vs_correct"] = ss.spearmanr(p.sim_max_train, ok.astype(int)).correlation
    filas.append(fila)
R5 = pd.DataFrame(filas); print(R5.round(3).to_string()); R5.to_csv(f"{OUTD}/overlap_effect.csv", index=False)
print("Reading: leakage if acc_with_overlap is clearly higher than acc_without_overlap and p_fisher < 0.05.")

In [ ]:
# Cell 6b: 5b. effect of the overlap within each class (Mantel-Haenszel)
from statsmodels.stats.contingency_tables import StratifiedTable
print("== 5b. Overlap within classes ==")
print(pd.crosstab(T.label, T.solape_max > 0, margins=True).rename(columns={False: "without_overlap", True: "with_overlap"}))
filas = []
for m in OK:
    p = PRED[m].merge(T[["patch_id", "solape_max"]], on="patch_id")
    p["ok"] = (p.y_true == p.y_pred).astype(int); p["sol"] = (p.solape_max > 0).astype(int)
    fila, tablas = dict(model=m), []
    for c in range(4):
        q = p[p.y_true == c]; con, sin = q[q.sol == 1], q[q.sol == 0]
        fila[f"{c}_with"] = con.ok.mean() if len(con) else np.nan
        fila[f"{c}_without"] = sin.ok.mean() if len(sin) else np.nan
        t = np.array([[con.ok.sum(), len(con) - con.ok.sum()], [sin.ok.sum(), len(sin) - sin.ok.sum()]], float)
        if len(con) and len(sin): tablas.append(t)
    if tablas:
        st = StratifiedTable(tablas)
        fila["OR_MH"] = st.oddsratio_pooled; fila["p_MH"] = st.test_null_odds(correction=True).pvalue
    filas.append(fila)
R5b = pd.DataFrame(filas); print(R5b.round(3).to_string()); R5b.to_csv(f"{OUTD}/overlap_effect_by_class.csv", index=False)
print("Reading: OR_MH > 1 with p_MH < 0.05 means higher accuracy on overlapping patches within the same class (leakage).")

In [ ]:
# Cell 7: 6. accuracy by range of loss_pct (class thresholds at 0, 3 and 7 %)
print("== 6. Accuracy by loss_pct range ==")
cortes = [-1e-9, 1e-9, 0.5, 2, 3, 4, 6, 7, 8, 12, 100]
nombres = ["0", "0-0.5", "0.5-2", "2-3", "3-4", "4-6", "6-7", "7-8", "8-12", ">12"]
tab = {}
for m in OK:
    p = PRED[m].merge(meta[["patch_id", "loss_pct"]], on="patch_id")
    p["ok"] = p.y_true == p.y_pred; p["rango"] = pd.cut(p.loss_pct, cortes, labels=nombres)
    tab[m] = p.groupby("rango", observed=False).ok.mean()
n = PRED[OK[0]].merge(meta[["patch_id", "loss_pct"]], on="patch_id")
R6 = pd.DataFrame(tab); R6.insert(0, "n_patches", pd.cut(n.loss_pct, cortes, labels=nombres).value_counts().reindex(nombres).values)
R6.index.name = "range"
print(R6.round(2).to_string()); R6.to_csv(f"{OUTD}/accuracy_by_loss_range.csv")
ax = R6.drop(columns="n_patches").plot(marker="o", figsize=(10, 4)); ax.set_ylabel("Accuracy"); ax.set_xlabel("loss_pct (%)")
for x in [0.5, 3.5, 6.5]: ax.axvline(x, color="gray", ls="--", lw=0.8)
plt.tight_layout(); plt.savefig(f"{OUTD}/accuracy_by_loss_range.png", dpi=200); plt.show()
print("Reading: if accuracy drops near 0-0.5, 2-4 and 6-8, the errors come from the class thresholds.")

In [ ]:
# Cell 8: 7. ordinal metrics and grouped formulations (without retraining)
print("== 7. Ordinal metrics ==")
a3 = {0: 0, 1: 1, 2: 1, 3: 2}; a2 = {0: 0, 1: 1, 2: 1, 3: 1}
filas = []
for m in OK:
    y, yh = PRED[m].y_true, PRED[m].y_pred
    filas.append(dict(model=m, acc_4_classes=accuracy_score(y, yh), f1_macro=f1_score(y, yh, average="macro"),
        acc_within_1=(abs(y - yh) <= 1).mean(), mae_classes=mean_absolute_error(y, yh),
        quadratic_kappa=cohen_kappa_score(y, yh, weights="quadratic"),
        acc_3_groups=accuracy_score(y.map(a3), yh.map(a3)), acc_2_groups=accuracy_score(y.map(a2), yh.map(a2))))
R7 = pd.DataFrame(filas); print(R7.round(3).to_string()); R7.to_csv(f"{OUTD}/ordinal_metrics_audit.csv", index=False)
print("3 groups: no deforestation | low + moderate | severe. 2 groups: no loss vs loss.")

In [ ]:
# Cell 9: 8. information in the data: correlations and classical baseline
print("== 8. Separability and baseline ==")
for c in ["dndvi_medio", "frac_caida_ndvi", "frac_suelo_2024", "ndvi_min_serie"]:
    print(f"  Spearman loss_pct vs {c:16s}: rho = {ss.spearmanr(meta.loss_pct, F[c]).correlation:.3f}")
y = meta.label.values; itr, ite = meta.split.values == "train", meta.split.values == "test"
filas = []
for nombre, clf in [("Logistic regression", make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000))),
                    ("Random Forest", RandomForestClassifier(500, random_state=42, n_jobs=-1))]:
    clf.fit(F.values[itr], y[itr]); yh = clf.predict(F.values[ite])
    filas.append(dict(model=nombre, acc=accuracy_score(y[ite], yh), f1_macro=f1_score(y[ite], yh, average="macro"),
                      acc_within_1=(abs(y[ite] - yh) <= 1).mean()))
R8 = pd.DataFrame(filas); print(R8.round(3).to_string()); R8.to_csv(f"{OUTD}/baseline.csv", index=False)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
for k, c in enumerate(["frac_caida_ndvi", "dndvi_medio"]):
    ax[k].boxplot([F.loc[meta.label == j, c] for j in range(4)]); ax[k].set_xticks(range(1, 5), CL); ax[k].set_title(c)
plt.tight_layout(); plt.savefig(f"{OUTD}/separability.png", dpi=200); plt.show()
print("Reading: if a Random Forest with simple descriptors approaches the best deep model, the limit is in the data and labels.")

In [ ]:
# Cell 10: 9. image gaps (masked clouds) and errors
print("== 9. Gaps and errors ==")
cal["vacio_max"] = cal[[f"vacio_{a}" for a in range(2020, 2025)]].max(axis=1)
tab = {}
for m in OK:
    p = PRED[m].merge(cal[["patch_id", "vacio_max"]], on="patch_id"); p["ok"] = p.y_true == p.y_pred
    tab[m] = p.groupby(pd.cut(p.vacio_max, [-0.01, 0.01, 0.1, 0.5, 1.0]), observed=False).ok.mean()
n = PRED[OK[0]].merge(cal[["patch_id", "vacio_max"]], on="patch_id")
R9 = pd.DataFrame(tab); R9.insert(0, "n_patches", pd.cut(n.vacio_max, [-0.01, 0.01, 0.1, 0.5, 1.0]).value_counts().sort_index().values)
print(R9.round(2).to_string())

In [ ]:
# Cell 11: 10. training dynamics (overfitting and underfitting)
print("== 10. Training ==")
filas = []
for m in OK:
    if m not in HIST: continue
    h = HIST[m]; b = h.val_loss.idxmin()
    filas.append(dict(model=m, epochs=len(h), best_epoch=int(h.epoch[b]), train_acc_best=h.train_acc[b],
                      val_acc_best=h.val_acc[b], train_acc_final=h.train_acc.iloc[-1],
                      final_gap=h.train_acc.iloc[-1] - h.val_acc.iloc[-1],
                      stopped_at_limit=len(h) >= 50 and b >= len(h) - 5))
R10 = pd.DataFrame(filas); print(R10.round(3).to_string())
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
for m in OK:
    if m in HIST: ax[0].plot(HIST[m].epoch, HIST[m].val_loss, label=m); ax[1].plot(HIST[m].epoch, HIST[m].train_acc - HIST[m].val_acc, label=m)
ax[0].set_title("Validation loss"); ax[1].set_title("Train acc - val acc (overfitting)")
for a in ax: a.set_xlabel("Epoch"); a.legend(fontsize=7)
plt.tight_layout(); plt.savefig(f"{OUTD}/training_dynamics.png", dpi=200); plt.show()

In [ ]:
# Cell 12: 11. agreement between models and patches misclassified by all models
print("== 11. Agreement between models ==")
A = pd.DataFrame({m: PRED[m].set_index("patch_id").sort_index().y_pred for m in OK})
yt = PRED[OK[0]].set_index("patch_id").sort_index().y_true
acierto = A.eq(yt, axis=0)
print("patches misclassified by ALL models:", int((~acierto).all(axis=1).sum()), "of", len(A))
print("patches classified correctly by at least one model:", round(float(acierto.any(axis=1).mean()), 3))
print("accuracy of the majority vote:", round(float((A.mode(axis=1)[0] == yt).mean()), 3))
dif = meta.set_index("patch_id").loc[acierto.index[(~acierto).all(axis=1)], ["label", "loss_pct", "sim_max_train"]]
dif = dif.join(A[(~acierto).all(axis=1)].add_prefix("pred_"))
print(dif.sort_values("loss_pct").round(3).to_string()); dif.to_csv(f"{OUTD}/hard_patches.csv")

In [ ]:
# Cell 13: 12. accuracy by geographic zone
print("== 12. Accuracy by zone ==")
cx = (meta.lon_min + meta.lon_max) / 2; cy = (meta.lat_min + meta.lat_max) / 2
meta["zona"] = np.select([(cx > -69.6) & (cy > -10.9), cx > -70.5, cx > -72.0],
                         ["Northeast (Acre and Pando)", "East", "Center"], "West")
tab = {}
for m in OK:
    p = PRED[m].merge(meta[["patch_id", "zona"]], on="patch_id"); tab[m] = p.assign(ok=p.y_true == p.y_pred).groupby("zona").ok.mean()
n = PRED[OK[0]].merge(meta[["patch_id", "zona"]], on="patch_id").zona.value_counts()
R12 = pd.DataFrame(tab); R12.insert(0, "n_patches", n.reindex(R12.index).values); print(R12.round(2).to_string())

In [ ]:
# Cell 14: 13. summary
print("== 13. SUMMARY ==")
print("Comparable models:", OK, "| excluded:", [m for m in PRED if m not in OK])
print("Identical images:", len(dup), "| test patches with similarity > 0.95 with training:", int((sim.max(1) > 0.95).sum()))
print("Test patches overlapping with training:", int((T.solape_max > 0).sum()), "of", len(T))
print(R5[["model", "acc_with_overlap", "acc_without_overlap"] + (["p_fisher"] if "p_fisher" in R5 else [])].round(3).to_string())
print(R7[["model", "acc_4_classes", "acc_within_1", "acc_3_groups", "acc_2_groups", "quadratic_kappa"]].round(3).to_string())
print(R8.round(3).to_string())
print("Files saved in", OUTD, ":", os.listdir(OUTD))

In [ ]:
# Cell 15: three formulations of the Random Forest (training and validation only, the test subset is not used)
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import StratifiedKFold, cross_val_predict
print("== Formulations (without the test subset) ==")
Xf, lab, z = F.values, meta.label.values, np.log1p(meta.loss_pct.values)
itr, iva = meta.split.values == "train", meta.split.values == "val"
cv = StratifiedKFold(5, shuffle=True, random_state=42)
rf = lambda: RandomForestClassifier(500, random_state=42, n_jobs=-1)

def ordinal(Xa, ya, Xb):                     # three classifiers "y >= k"; class = number of them above 0.5
    return sum((rf().fit(Xa, (ya >= k).astype(int)).predict_proba(Xb)[:, 1] > .5).astype(int) for k in (1, 2, 3))

def mejores_cortes(zhat, y):                 # cut points on the predicted loss that maximize the training accuracy
    g = np.unique(np.quantile(zhat, np.linspace(.02, .98, 49))); best = (-1, None)
    for i, a in enumerate(g):
        for j, b in enumerate(g[i + 1:], i + 1):
            for c in g[j + 1:]:
                acc = (np.digitize(zhat, [a, b, c]) == y).mean()
                if acc > best[0]: best = (acc, [a, b, c])
    return best[1]

res, oof = [], {}
Xt, yt, zt = Xf[itr], lab[itr], z[itr]
folds = list(cv.split(Xt, yt))               # the same five folds for the three formulations
oof["a) four-class"] = cross_val_predict(rf(), Xt, yt, cv=folds)
va_a = rf().fit(Xt, yt).predict(Xf[iva])
oof_b = np.zeros(len(yt), int)
for f_tr, f_te in folds: oof_b[f_te] = ordinal(Xt[f_tr], yt[f_tr], Xt[f_te])
oof["b) ordinal"] = oof_b; va_b = ordinal(Xt, yt, Xf[iva])
reg = lambda: RandomForestRegressor(500, random_state=42, n_jobs=-1)
zoof = cross_val_predict(reg(), Xt, zt, cv=folds); cortes = mejores_cortes(zoof, yt)
oof["c) regression"] = np.digitize(zoof, cortes)
va_c = np.digitize(reg().fit(Xt, zt).predict(Xf[iva]), cortes)
print("learned cut points (loss_pct %):", np.round(np.expm1(cortes), 3))
for nombre, va in zip(oof, [va_a, va_b, va_c]):
    o = oof[nombre]
    res.append(dict(formulation=nombre, acc_cv_train=accuracy_score(yt, o), f1_cv_train=f1_score(yt, o, average="macro"),
                    acc_val=accuracy_score(lab[iva], va), f1_val=f1_score(lab[iva], va, average="macro"),
                    **{f"recall_val_{c}": (va[lab[iva] == c] == c).mean() for c in range(4)}))
R15 = pd.DataFrame(res); print(R15.round(3).to_string()); R15.to_csv(f"{OUTD}/formulations.csv", index=False)